# Shortest path


In [11]:
import networkx as nx
import spacy
from collections import deque

# (The find_shortest_path function from the previous answer remains unchanged)
def find_shortest_path(G: nx.MultiDiGraph, start_word: str, end_word: str, M: int):
    """
    Finds the shortest path between two nodes in a NetworkX MultiDiGraph based on
    their 'word' attribute, with a maximum number M of intermediate nodes.
    """
    # 1. Find the node identifiers for the start and end words
    start_node, end_node = None, None
    # Using a dictionary for efficient lookup of the *first* node with a given word
    node_word_map = {data['word']: node for node, data in reversed(list(G.nodes(data=True)))}

    start_node = node_word_map.get(start_word)
    end_node = node_word_map.get(end_word)

    if start_node is None or end_node is None:
        print(f"⚠️ Error: Start word '{start_word}' or end word '{end_word}' not found.")
        return None

    if start_node == end_node:
        return [start_word]

    # 2. Initialize BFS
    queue = deque([(start_node, [start_node])])
    visited = {start_node}

    # 3. Perform BFS
    while queue:
        current_node, path = queue.popleft()

        for neighbor in G.neighbors(current_node):
            if neighbor not in visited:
                new_path = path + [neighbor]
                if neighbor == end_node:
                    num_intermediate = len(new_path) - 2
                    if num_intermediate <= M:
                        return [G.nodes[n]['word'] for n in new_path]
                    else:
                        #print(f"Path found, but it has {num_intermediate} intermediate nodes (M={M} allowed).")
                        return None

                visited.add(neighbor)
                queue.append((neighbor, new_path))
    return None

# --- New Function to Build Graph using spaCy ---

def create_graph_from_sentence(nlp_doc) -> nx.MultiDiGraph:
    """
    Creates a MultiDiGraph from a spaCy Doc object.

    Nodes are tokens, and edges represent both sequence and dependency relations.
    """
    G = nx.MultiDiGraph()
    for token in nlp_doc:
        # Add each token as a node
        G.add_node(token.i, word=token.text)

    for token in nlp_doc:
        # Add sequence edges (word -> next word)
        if token.i + 1 < len(nlp_doc):
            G.add_edge(token.i, token.i + 1, type='seq')

        # Add dependency edges (head -> child)
        if token.head.i != token.i: # Exclude self-loops for root
            G.add_edge(token.head.i, token.i, type='dep', label=token.dep_)

    return G

# Expected: None

In [12]:

# --- Main Test Code ---

# 1. Load the spaCy model
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")
print("Model loaded.")

# 2. Process a sentence and create the graph
sentence = "The quick brown fox jumps over the lazy dog."
doc = nlp(sentence)
G = create_graph_from_sentence(doc)

print(f"\nGraph created from sentence: '{sentence}'")

# --- Test Cases ---

# Test Case 1: Dependency path is the shortest
# The dependency path from 'jumps' (verb) to 'dog' (object of preposition)
# is 'jumps' -> 'over' -> 'dog'. This is shorter than the sequence path.
print("\nTest Case 1: Searching from 'jumps' to 'dog' with M=1")
path1 = find_shortest_path(G, "jumps", "dog", M=1)
print(f"Path found: {path1}")
# Expected: ['jumps', 'over', 'dog'] because it has 1 intermediate node.

print("---")

# Test Case 2: Sequence path is the only option
# There is no direct dependency path from 'quick' to 'jumps'.
print("\n🔎 Test Case 2: Searching from 'quick' to 'jumps' with M=2")
path2 = find_shortest_path(G, "quick", "jumps", M=2)
print(f"✅ Path found: {path2}")
# Expected: ['quick', 'brown', 'fox', 'jumps']

print("---")

# Test Case 3: Shortest path exists but violates the constraint
# The shortest path from 'jumps' to 'dog' has 1 intermediate node ('over').
# If we only allow M=0, no path should be returned.
print("\n🔎 Test Case 3: Searching from 'jumps' to 'dog' with M=0")
path3 = find_shortest_path(G, "jumps", "dog", M=0)
print(f"✅ Path found: {path3}")

Loading spaCy model...
Model loaded.

Graph created from sentence: 'The quick brown fox jumps over the lazy dog.'

Test Case 1: Searching from 'jumps' to 'dog' with M=1
Path found: ['jumps', 'over', 'dog']
---

🔎 Test Case 2: Searching from 'quick' to 'jumps' with M=2
✅ Path found: ['quick', 'brown', 'fox', 'jumps']
---

🔎 Test Case 3: Searching from 'jumps' to 'dog' with M=0
✅ Path found: None


In [13]:
import networkx as nx
import spacy
from collections import deque
import time
import random


def run_stress_test(G: nx.MultiDiGraph, num_iterations: int):
    """
    Performs a stress test on the find_shortest_path function.

    Args:
        G (nx.MultiDiGraph): The graph to search on.
        num_iterations (int): The number of random searches to perform.
    """
    print("\n🚀 Starting Stress Test...")

    words_in_graph = [data['word'] for _, data in G.nodes(data=True)]
    max_M = len(words_in_graph) - 2

    paths_found = 0
    paths_not_found = 0

    # Start the timer
    start_time = time.perf_counter()

    for i in range(num_iterations):
        # Select two different random words
        start_word, end_word = random.sample(words_in_graph, 2)

        # Select a random M
        M = random.randint(0, max_M)

        path = find_shortest_path(G, start_word, end_word, M)

        if path:
            paths_found += 1
        else:
            paths_not_found += 1

    # Stop the timer
    end_time = time.perf_counter()
    total_time = end_time - start_time

    # --- Generate Report ---
    print("Stress Test Complete. Generating report...\n")
    print("="*40)
    print("📊 PERFORMANCE REPORT")
    print("="*40)

    print("\n## Test Configuration")
    print(f"  - Total Searches Performed: {num_iterations:,}")
    print(f"  - Graph Nodes (Words):      {G.number_of_nodes()}")
    print(f"  - Graph Edges (Relations):  {G.number_of_edges()}")

    print("\n## Execution Time")
    print(f"  - Total Execution Time: {total_time:.4f} seconds")
    avg_time_ms = (total_time / num_iterations) * 1000
    avg_time_us = avg_time_ms * 1000
    print(f"  - Average Time per Search: {avg_time_us:.2f} µs (microseconds)")

    print("\n## Search Results")
    print(f"  - Paths Found:           {paths_found:,} ({paths_found/num_iterations:.2%})")
    print(f"  - Paths Not Found:       {paths_not_found:,} ({paths_not_found/num_iterations:.2%})")
    print("="*40)


# --- Main Execution ---

if __name__ == "__main__":
    # 1. Load the spaCy model
    print("Loading spaCy model...")
    nlp = spacy.load("en_core_web_lg")

    # 2. Use a complex sentence for a more realistic test
    sentence = """
It has been suggested that legal expert systems could help to manage the rapid expansion of legal information and decisions that began to intensify in the late 1960s.[2] Many of the first legal expert systems were created in the 1970s[1]: and 1980s.[3]:

Lawyers were originally identified as primary target users of legal expert systems.[4]: Potential motivations for this work included:

quicker delivery of legal advice;
reduced time spent in repetitive, labour-intensive legal tasks;
development of knowledge management techniques that were not dependent on staff;
reduced overhead and labour costs and higher profitability for law firms; and
reduced fees for clients.[5]:4
Some early development work was oriented toward the creation of automated judges.[6]:

One of the first use cases was the encoding of the British Nationality Act at Imperial College carried out under the supervision of Marek Sergot and Robert Kowalski. Lance Elliot wrote: "The British Nationality Act was passed in 1981 and shortly thereafter was used as a means of showcasing the efficacy of using Artificial Intelligence (AI) techniques and technologies, doing so to explore how the at-the-time newly enacted statutory law might be encoded into a computerized logic-based formalization." [7]

The authors’ seminal article, "The British Nationality Act as a Logic Program," published in 1986 in the Communications of the ACM journal, is one of the first and best-known works in computational law, and one of the most widely cited papers in the field.[8]

In 2021, the Inaugural CodeX Prize was awarded to Robert Kowalski, Fariba Sadri, and Marek Sergot in acknowledgment of their groundbreaking work on the application of logic programming to the formalization and analysis of the British Nationality Act.[9]

Later work on legal expert systems has identified potential benefits to non-lawyers as a means to increase access to legal knowledge.[4]:

Legal expert systems can also support administrative processes, facilitate decision-making processes, automate rule-based analyses,[10] and exchange information directly with citizen-users.[11]
    """
    doc = nlp(sentence)
    G = create_graph_from_sentence(doc)

    # 3. Run the test
    run_stress_test(G, num_iterations=1_000_000)

Loading spaCy model...

🚀 Starting Stress Test...
Stress Test Complete. Generating report...

📊 PERFORMANCE REPORT

## Test Configuration
  - Total Searches Performed: 1,000,000
  - Graph Nodes (Words):      389
  - Graph Edges (Relations):  772

## Execution Time
  - Total Execution Time: 93.2525 seconds
  - Average Time per Search: 93.25 µs (microseconds)

## Search Results
  - Paths Found:           608,900 (60.89%)
  - Paths Not Found:       391,100 (39.11%)


In [14]:
import networkx as nx
import spacy
from collections import deque
from itertools import permutations
import math

# --- (Previous functions: find_shortest_path, create_graph_from_sentence) ---
# These functions are required as building blocks.

def find_shortest_path(G: nx.MultiDiGraph, start_word: str, end_word: str, M: int):
    start_node, end_node = None, None
    node_word_map = {data['word']: node for node, data in reversed(list(G.nodes(data=True)))}
    start_node = node_word_map.get(start_word)
    end_node = node_word_map.get(end_word)
    if start_node is None or end_node is None: return None
    if start_node == end_node: return [start_word]
    queue = deque([(start_node, [start_node])])
    visited = {start_node}
    while queue:
        current_node, path = queue.popleft()
        for neighbor in G.neighbors(current_node):
            if neighbor not in visited:
                new_path = path + [neighbor]
                if neighbor == end_node:
                    if (len(new_path) - 2) <= M: return [G.nodes[n]['word'] for n in new_path]
                    else: return None
                visited.add(neighbor)
                queue.append((neighbor, new_path))
    return None

def create_graph_from_sentence(nlp_doc) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()
    for token in nlp_doc: G.add_node(token.i, word=token.text)
    for token in nlp_doc:
        if token.i + 1 < len(nlp_doc): G.add_edge(token.i, token.i + 1, type='seq')
        if token.head.i != token.i: G.add_edge(token.head.i, token.i, type='dep', label=token.dep_)
    return G

# --- New Algorithm Implementation ---

def find_path_connecting_nodes(G: nx.MultiDiGraph, words: list, M: int):
    """
    Finds the shortest continuous path connecting a list of nodes in any order.

    Args:
        G (nx.MultiDiGraph): The graph to search on.
        words (list): A list of three word strings to connect.
        M (int): The max number of intermediate nodes allowed for each path segment.

    Returns:
        list[str]: The shortest combined path as a list of words.
        None: If no valid path connecting all nodes is found.
    """
    if len(words) != 3:
        raise ValueError("This implementation requires a list of exactly three words.")

    shortest_overall_path = None
    min_length = float('inf')

    # 1. Iterate through all possible orderings of the words
    for p in permutations(words):
        # Permutation is a tuple, e.g., ('fox', 'dog', 'quick')
        node1, node2, node3 = p[0], p[1], p[2]

        # 2. Find the path for the first and second segments
        path1 = find_shortest_path(G, node1, node2, M)
        if path1 is None:
            continue # If the first segment fails, this permutation is impossible

        path2 = find_shortest_path(G, node2, node3, M)
        if path2 is None:
            continue # If the second segment fails, this permutation is impossible

        # 3. Combine the paths and check if it's the shortest found so far
        # Use path2[1:] to avoid duplicating the middle node
        combined_path = path1 + path2[1:]

        if len(combined_path) < min_length:
            min_length = len(combined_path)
            shortest_overall_path = combined_path

    return shortest_overall_path

# --- Main Test Code ---

if __name__ == "__main__":
    # 1. Load spaCy and create a graph from a sentence
    nlp = spacy.load("en_core_web_lg")
    sentence = "The quick brown fox jumps over the lazy dog."
    doc = nlp(sentence)
    G = create_graph_from_sentence(doc)

    print(f"📝 Graph created from sentence: '{sentence}'")

    # 2. Define the three words to connect and the constraint
    nodes_to_connect = ['fox', 'dog', 'quick']
    M_constraint = 1

    print(f"\n🔎 Searching for the shortest path connecting {nodes_to_connect}...")
    print(f"   (Constraint: Max {M_constraint} intermediate nodes per segment)")

    # 3. Run the algorithm
    final_path = find_path_connecting_nodes(G, nodes_to_connect, M=M_constraint)

    # 4. Report the result
    if final_path:
        print("\n✅ Path found!")
        print(f"   - Order: {final_path[0]} → ... → {final_path[-1]}")
        print(f"   - Path: {' → '.join(final_path)}")
    else:
        print("\n❌ No valid path could be found connecting all nodes with the given constraints.")

📝 Graph created from sentence: 'The quick brown fox jumps over the lazy dog.'

🔎 Searching for the shortest path connecting ['fox', 'dog', 'quick']...
   (Constraint: Max 1 intermediate nodes per segment)

❌ No valid path could be found connecting all nodes with the given constraints.


In [15]:
import networkx as nx
import spacy
from collections import deque
from itertools import permutations
import time
import random

# --- Required Functions (from previous answers) ---

def find_shortest_path(G: nx.MultiDiGraph, start_word: str, end_word: str, M: int):
    start_node, end_node = None, None
    node_word_map = {data['word']: node for node, data in reversed(list(G.nodes(data=True)))}
    start_node = node_word_map.get(start_word)
    end_node = node_word_map.get(end_word)
    if start_node is None or end_node is None: return None
    if start_node == end_node: return [start_word]
    queue = deque([(start_node, [start_node])])
    visited = {start_node}
    while queue:
        current_node, path = queue.popleft()
        for neighbor in G.neighbors(current_node):
            if neighbor not in visited:
                new_path = path + [neighbor]
                if neighbor == end_node:
                    if (len(new_path) - 2) <= M: return [G.nodes[n]['word'] for n in new_path]
                    else: return None
                visited.add(neighbor)
                queue.append((neighbor, new_path))
    return None

def create_graph_from_sentence(nlp_doc) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()
    for token in nlp_doc: G.add_node(token.i, word=token.text)
    for token in nlp_doc:
        if token.i + 1 < len(nlp_doc): G.add_edge(token.i, token.i + 1, type='seq')
        if token.head.i != token.i: G.add_edge(token.head.i, token.i, type='dep', label=token.dep_)
    return G

def find_path_connecting_nodes(G: nx.MultiDiGraph, words: list, M: int):
    if len(words) != 3: raise ValueError("This implementation requires a list of exactly three words.")
    shortest_overall_path = None
    min_length = float('inf')
    for p in permutations(words):
        path1 = find_shortest_path(G, p[0], p[1], M)
        if path1 is None: continue
        path2 = find_shortest_path(G, p[1], p[2], M)
        if path2 is None: continue
        combined_path = path1 + path2[1:]
        if len(combined_path) < min_length:
            min_length = len(combined_path)
            shortest_overall_path = combined_path
    return shortest_overall_path

# --- New Stress Test Implementation ---

def run_multinode_stress_test(G: nx.MultiDiGraph, num_iterations: int):
    """
    Performs a stress test on the find_path_connecting_nodes function.
    """
    print("\n🚀 Starting Multi-Node Stress Test...")

    words_in_graph = list(set([data['word'] for _, data in G.nodes(data=True)]))
    max_M = len(words_in_graph) - 2

    paths_found = 0
    paths_not_found = 0

    start_time = time.perf_counter()

    for _ in range(num_iterations):
        # Select three different random words
        words_to_connect = random.sample(words_in_graph, 3)
        M = random.randint(0, max_M)

        path = find_path_connecting_nodes(G, words_to_connect, M)

        if path:
            paths_found += 1
        else:
            paths_not_found += 1

    end_time = time.perf_counter()
    total_time = end_time - start_time

    # --- Generate Report ---
    print("Stress Test Complete. Generating report...\n")
    print("="*40)
    print("📊 MULTI-NODE PERFORMANCE REPORT")
    print("="*40)
    print("\n## Test Configuration")
    print(f"  - Function Tested:          find_path_connecting_nodes")
    print(f"  - Total Searches Performed: {num_iterations:,}")
    print(f"  - Graph Nodes (Words):      {G.number_of_nodes()}")
    print(f"  - Graph Edges (Relations):  {G.number_of_edges()}")

    print("\n## Execution Time")
    print(f"  - Total Execution Time:     {total_time:.4f} seconds")
    avg_time_ms = (total_time / num_iterations) * 1000
    print(f"  - Average Time per Search:  {avg_time_ms:.2f} ms (milliseconds)")

    print("\n## Search Results")
    print(f"  - Paths Found:              {paths_found:,} ({paths_found/num_iterations:.2%})")
    print(f"  - Paths Not Found:          {paths_not_found:,} ({paths_not_found/num_iterations:.2%})")
    print("="*40)


# --- Main Execution ---
if __name__ == "__main__":
    nlp = spacy.load("en_core_web_lg")
    sentence = """
One of the first use cases was the encoding of the British Nationality Act at Imperial College carried out under the supervision of Marek Sergot and Robert Kowalski. Lance Elliot wrote: "The British Nationality Act was passed in 1981 and shortly thereafter was used as a means of showcasing the efficacy of using Artificial Intelligence (AI) techniques and technologies, doing so to explore how the at-the-time newly enacted statutory law might be encoded into a computerized logic-based formalization."
    """
    doc = nlp(sentence)
    G = create_graph_from_sentence(doc)

    # Run the new stress test with 100,000 iterations
    run_multinode_stress_test(G, num_iterations=100_000)


🚀 Starting Multi-Node Stress Test...
Stress Test Complete. Generating report...

📊 MULTI-NODE PERFORMANCE REPORT

## Test Configuration
  - Function Tested:          find_path_connecting_nodes
  - Total Searches Performed: 100,000
  - Graph Nodes (Words):      94
  - Graph Edges (Relations):  184

## Execution Time
  - Total Execution Time:     18.0200 seconds
  - Average Time per Search:  0.18 ms (milliseconds)

## Search Results
  - Paths Found:              83,307 (83.31%)
  - Paths Not Found:          16,693 (16.69%)


In [ ]:
import networkx as nx
import spacy
from collections import deque
from itertools import permutations

# --- Required Functions (from previous answers with modifications) ---

def find_shortest_path(G: nx.MultiDiGraph, start_word: str, end_word: str, M: int, edge_type: str = 'both'):
    """
    Finds the shortest path between two nodes using specified edge types.

    Args:
        G (nx.MultiDiGraph): The graph to search on.
        start_word (str): The word of the starting node.
        end_word (str): The word of the ending node.
        M (int): The max number of intermediate nodes allowed.
        edge_type (str): The type of edge to use: 'seq', 'dep', or 'both'.
                         Defaults to 'both'.
    """
    if edge_type not in ['seq', 'dep', 'both']:
        raise ValueError("edge_type must be 'seq', 'dep', or 'both'.")

    # Node lookup (remains the same)
    start_node, end_node = None, None
    node_word_map = {data['word']: node for node, data in reversed(list(G.nodes(data=True)))}
    start_node = node_word_map.get(start_word)
    end_node = node_word_map.get(end_word)
    if start_node is None or end_node is None: return None
    if start_node == end_node: return [start_word]

    # BFS Initialization (remains the same)
    queue = deque([(start_node, [start_node])])
    visited = {start_node}

    while queue:
        current_node, path = queue.popleft()

        # --- MODIFIED NEIGHBOR EXPLORATION ---
        # Iterate through outgoing edges from the current node
        for u, v, data in G.edges(current_node, data=True):
            # Check if the edge type matches the search criteria
            use_edge = (edge_type == 'both' or data.get('type') == edge_type)

            if use_edge and v not in visited:
                new_path = path + [v]
                if v == end_node:
                    if (len(new_path) - 2) <= M:
                        return [G.nodes[n]['word'] for n in new_path]
                    else:
                        return None

                visited.add(v)
                queue.append((v, new_path))

    return None


def find_path_connecting_nodes(G: nx.MultiDiGraph, words: list, M: int, edge_type: str = 'both'):
    """
    Finds the shortest path connecting three nodes, propagating the edge_type parameter.
    """
    if len(words) != 3: raise ValueError("This implementation requires a list of exactly three words.")
    shortest_overall_path = None
    min_length = float('inf')

    for p in permutations(words):
        # Pass the edge_type parameter to the sub-routine
        path1 = find_shortest_path(G, p[0], p[1], M, edge_type)
        if path1 is None: continue

        path2 = find_shortest_path(G, p[1], p[2], M, edge_type)
        if path2 is None: continue

        combined_path = path1 + path2[1:]
        if len(combined_path) < min_length:
            min_length = len(combined_path)
            shortest_overall_path = combined_path

    return shortest_overall_path

# Helper function from previous steps
def create_graph_from_sentence(nlp_doc) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()
    for token in nlp_doc: G.add_node(token.i, word=token.text)
    for token in nlp_doc:
        if token.i + 1 < len(nlp_doc): G.add_edge(token.i, token.i + 1, type='seq')
        if token.head.i != token.i: G.add_edge(token.head.i, token.i, type='dep', label=token.dep_)
    return G

# --- Main Test Code ---

if __name__ == "__main__":
    nlp = spacy.load("en_core_web_sm")
    sentence = "The quick brown fox jumps over the lazy dog."
    doc = nlp(sentence)
    G = create_graph_from_sentence(doc)

    print(f"📝 Graph created from sentence: '{sentence}'")
    print("-" * 50)

    # --- Test Cases ---

    print("🔎 Test Case 1: Dependency-only path")
    print("   Searching from 'jumps' to 'fox' (head to subject). This path only exists via 'dep' edge.")
    path_dep = find_shortest_path(G, "jumps", "fox", M=1, edge_type='dep')
    print(f"   - Result with edge_type='dep': {path_dep}")
    path_seq = find_shortest_path(G, "jumps", "fox", M=1, edge_type='seq')
    print(f"   - Result with edge_type='seq': {path_seq}")
    print("-" * 50)


    print("🔎 Test Case 2: Sequence-only path")
    print("   Searching from 'brown' to 'fox'. This is a direct sequential link.")
    path_seq_2 = find_shortest_path(G, "brown", "fox", M=1, edge_type='seq')
    print(f"   - Result with edge_type='seq': {path_seq_2}")
    path_dep_2 = find_shortest_path(G, "brown", "fox", M=1, edge_type='dep')
    print(f"   - Result with edge_type='dep': {path_dep_2}") # 'brown' is a modifier of 'fox'
    print("-" * 50)


    print("🔎 Test Case 3: 'both' vs specific types")
    print("   Searching from 'jumps' to 'dog'. The shortest path uses dependency links.")
    path_both = find_shortest_path(G, "jumps", "dog", M=4, edge_type='both')
    print(f"   - Result with edge_type='both': {path_both} (Length: {len(path_both) if path_both else 0})")
    path_seq_3 = find_shortest_path(G, "jumps", "dog", M=4, edge_type='seq')
    print(f"   - Result with edge_type='seq':  {path_seq_3} (Length: {len(path_seq_3) if path_seq_3 else 0})")
    print("-" * 50)

    print("🔎 Test Case 4: Multi-node search with specific edge types")
    print("   Connecting ('jumps', 'fox', 'quick') using only dependency edges.")
    path_multi = find_path_connecting_nodes(G, ['jumps', 'fox', 'quick'], M=2, edge_type='dep')
    print(f"   - Result with edge_type='dep': {' → '.join(path_multi) if path_multi else 'None'}")
    print("-" * 50)

In [ ]:
def find_shortest_path(G: nx.MultiDiGraph, start_word: str, end_word: str, M: int, edge_type: str = 'any'):
    """
    Finds the shortest path with an edge type constraint.

    Args:
        G (nx.MultiDiGraph): The graph to search on.
        start_word (str): The word of the starting node.
        end_word (str): The word of the ending node.
        M (int): The maximum number of intermediate nodes allowed.
        edge_type (str): The type of edge to use ('seq', 'dep', or 'any').
    """
    start_node, end_node = None, None
    node_word_map = {data['word']: node for node, data in reversed(list(G.nodes(data=True)))}
    start_node = node_word_map.get(start_word)
    end_node = node_word_map.get(end_word)
    if start_node is None or end_node is None: return None
    if start_node == end_node: return [start_word]

    queue = deque([(start_node, [start_node])])
    visited = {start_node}

    while queue:
        current_node, path = queue.popleft()

        # --- MODIFIED NEIGHBOR EXPLORATION ---
        # Iterate through outgoing edges to inspect their type
        for u, v, data in G.edges(current_node, data=True):
            # Check if the edge type matches the search criteria
            if edge_type != 'any' and data.get('type') != edge_type:
                continue # Skip this edge if the type doesn't match

            neighbor = v
            if neighbor not in visited:
                new_path = path + [neighbor]
                if neighbor == end_node:
                    if (len(new_path) - 2) <= M:
                        return [G.nodes[n]['word'] for n in new_path]
                    else:
                        return None

                visited.add(neighbor)
                queue.append((neighbor, new_path))

    return None

In [ ]:
from itertools import permutations

def find_path_connecting_nodes(G: nx.MultiDiGraph, words: list, M: int, edge_type: str = 'any'):
    """
    Finds the shortest path connecting nodes with an edge type constraint.
    """
    if len(words) != 3: raise ValueError("This implementation requires a list of three words.")

    shortest_overall_path = None
    min_length = float('inf')

    for p in permutations(words):
        # Pass the edge_type parameter down to each call
        path1 = find_shortest_path(G, p[0], p[1], M, edge_type=edge_type)
        if path1 is None: continue

        path2 = find_shortest_path(G, p[1], p[2], M, edge_type=edge_type)
        if path2 is None: continue

        combined_path = path1 + path2[1:]
        if len(combined_path) < min_length:
            min_length = len(combined_path)
            shortest_overall_path = combined_path

    return shortest_overall_path

In [ ]:

if __name__ == "__main__":
    nlp = spacy.load("en_core_web_sm")
    sentence = "The quick brown fox jumps over the lazy dog."
    doc = nlp(sentence)
    G = create_graph_from_sentence(doc)

    print(f"📝 Graph created from sentence: '{sentence}'")
    start, end = "jumps", "dog"
    M = 4 # A reasonable constraint

    # Test 1: Search using ANY edge type (default behavior)
    print(f"\n🔎 Test 1: Searching from '{start}' to '{end}' using ANY edge type...")
    path_any = find_shortest_path(G, start, end, M, edge_type='any')
    print(f"   - Path Found: {' → '.join(path_any) if path_any else 'None'}")
    # Expected: ['jumps', 'over', 'dog'] (shortest path via dependency)

    # Test 2: Search using ONLY sequence edges
    print(f"\n🔎 Test 2: Searching from '{start}' to '{end}' using ONLY 'seq' edges...")
    path_seq = find_shortest_path(G, start, end, M, edge_type='seq')
    print(f"   - Path Found: {' → '.join(path_seq) if path_seq else 'None'}")
    # Expected: ['jumps', 'over', 'the', 'lazy', 'dog'] (must follow sentence order)

    # Test 3: Search using ONLY dependency edges
    print(f"\n🔎 Test 3: Searching from '{start}' to '{end}' using ONLY 'dep' edges...")
    path_dep = find_shortest_path(G, start, end, M, edge_type='dep')
    print(f"   - Path Found: {' → '.join(path_dep) if path_dep else 'None'}")
    # Expected: ['jumps', 'over', 'dog'] (same as 'any' in this case)
